In [0]:
cards_df = spark.table("jarvis_training.bronze.cards_data")
transactions_df = spark.table("jarvis_training.bronze.transactions_data")
user_df = spark.table("jarvis_training.bronze.users_data")
mcc_df = spark.table("jarvis_training.bronze.mcc_codes")
fraud_df = spark.table("jarvis_training.bronze.train_fraud_labels")

In [0]:
# cards_df transformation
cards_df.printSchema()
display(cards_df.limit(10))


root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |-- card_on_dark_web: string (nullable = true)



id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,true,2,33900.0000,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,true,1,11600.0000,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,true,1,19948.0000,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,true,2,16400.0000,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,true,2,19439.0000,01/1997,2007,No
5,619,Visa,Debit,4657824650820465,04/2024,245,true,2,21883.0000,01/1997,2012,No
6,1046,Amex,Credit,394584924614148,02/1999,302,true,2,9400.0000,01/1998,2011,No
7,511,Mastercard,Debit,5585238056278288,03/2005,749,true,1,9664.0000,01/1998,2011,No
8,1107,Mastercard,Credit,5462760953855576,09/2021,665,false,2,10300.0000,01/1998,2006,No
9,1046,Amex,Credit,357982644067712,09/2020,72,true,1,13000.0000,01/1999,2005,No


In [0]:
from pyspark.sql.functions import col, to_date
silver_cards_df = (

    cards_df
    .dropDuplicates(["id"])
    .withColumnRenamed("id", "card_id")
    .withColumn("credit_limit", col("credit_limit").cast("decimal(12,2)"))
    .withColumn("acct_open_date", to_date(col("acct_open_date"), "MM/yyyy"))
    .drop("card_number", "cvv")
)

display(silver_cards_df.limit(10))

card_id,client_id,card_brand,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,04/2024,true,2,33900.00,1991-01-01,2014,No
1,550,Mastercard,Credit,06/2024,true,1,11600.00,1994-01-01,2013,No
2,556,Mastercard,Debit,09/2021,true,1,19948.00,1995-01-01,2011,No
3,1937,Visa,Credit,04/2020,true,2,16400.00,1995-01-01,2015,No
4,1981,Mastercard,Debit,03/2024,true,2,19439.00,1997-01-01,2007,No
5,619,Visa,Debit,04/2024,true,2,21883.00,1997-01-01,2012,No
6,1046,Amex,Credit,02/1999,true,2,9400.00,1998-01-01,2011,No
7,511,Mastercard,Debit,03/2005,true,1,9664.00,1998-01-01,2011,No
8,1107,Mastercard,Credit,09/2021,false,2,10300.00,1998-01-01,2006,No
9,1046,Amex,Credit,09/2020,true,1,13000.00,1999-01-01,2005,No


In [0]:
silver_cards_df.write.mode("overwrite").saveAsTable("jarvis_training.silver.silver_cards")

In [0]:
# users_df transformation
user_df.printSchema()
display(user_df.limit(10))

root
 |-- id: string (nullable = true)
 |-- current_age: string (nullable = true)
 |-- retirement_age: string (nullable = true)
 |-- birth_year: string (nullable = true)
 |-- birth_month: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- num_credit_cards: string (nullable = true)



id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,$20599,$41997,$0,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,$25258,$51500,$102286,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,$26790,$54623,$114711,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,$26273,$42509,$2895,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,$18730,$38190,$81262,810,1


In [0]:
from pyspark.sql.functions import regexp_replace

silver_users_df = (
    user_df
    .dropDuplicates(["id"])
    .withColumnRenamed("id", "client_id")
    .withColumn(
        "per_capita_income",
        regexp_replace(col("per_capita_income"), "[$,]", "").cast("decimal(12,2)")
    )
    .withColumn(
        "yearly_income",
        regexp_replace(col("yearly_income"), "[$,]", "").cast("decimal(12,2)")
    )
    .withColumn(
        "total_debt",
        regexp_replace(col("total_debt"), "[$,]", "").cast("decimal(12,2)")
    )
)

display(silver_users_df.limit(10))
silver_users_df.printSchema()

client_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.00,59696.00,127613.00,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.00,77254.00,191349.00,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.00,33483.00,196.00,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.00,249925.00,202328.00,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.00,109687.00,183855.00,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,20599.00,41997.00,0.00,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,25258.00,51500.00,102286.00,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,26790.00,54623.00,114711.00,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,26273.00,42509.00,2895.00,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,18730.00,38190.00,81262.00,810,1


root
 |-- client_id: string (nullable = true)
 |-- current_age: string (nullable = true)
 |-- retirement_age: string (nullable = true)
 |-- birth_year: string (nullable = true)
 |-- birth_month: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- per_capita_income: decimal(12,2) (nullable = true)
 |-- yearly_income: decimal(12,2) (nullable = true)
 |-- total_debt: decimal(12,2) (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- num_credit_cards: string (nullable = true)



In [0]:
silver_users_df.write.mode("overwrite").saveAsTable("jarvis_training.silver.silver_users")

In [0]:
# transactions_df transformation
transactions_df.printSchema()
display(transactions_df.limit(10))

root
 |-- id: long (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: long (nullable = true)
 |-- card_id: long (nullable = true)
 |-- amount: string (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: long (nullable = true)
 |-- errors: string (nullable = true)



id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
10916267,2012-03-28T11:11:00.000Z,1238,1195,$175.73,Swipe Transaction,60569,Alexandria,VA,22309.0,5300,null
10916268,2012-03-28T11:12:00.000Z,148,5851,$20.29,Swipe Transaction,25717,Laconia,NH,3247.0,5812,null
10916269,2012-03-28T11:12:00.000Z,1433,5841,$17.80,Swipe Transaction,69972,Vicksburg,MS,39180.0,5814,null
10916270,2012-03-28T11:12:00.000Z,1753,5634,$3.73,Swipe Transaction,50783,Clinton,MS,39056.0,5411,null
10916271,2012-03-28T11:13:00.000Z,57,6006,$2.78,Swipe Transaction,14528,Tucson,AZ,85743.0,5499,null
10916272,2012-03-28T11:13:00.000Z,380,5838,$12.43,Swipe Transaction,75781,Des Moines,IA,50317.0,5411,null
10916273,2012-03-28T11:13:00.000Z,386,3673,$13.37,Online Transaction,16798,ONLINE,null,null,4121,null
10916274,2012-03-28T11:13:00.000Z,1446,3970,$58.00,Swipe Transaction,59935,Broken Bow,OK,74728.0,5499,null
10916276,2012-03-28T11:13:00.000Z,1662,4541,$114.97,Swipe Transaction,61709,Belleville,IL,62221.0,4900,null
10916277,2012-03-28T11:14:00.000Z,162,4266,$19.39,Swipe Transaction,39785,West Bloomfield,MI,48322.0,5912,null


In [0]:
silver_transactions_df = (
    transactions_df
    .dropDuplicates(["id"])
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("mcc", "mcc_code")
    .withColumn(
        "amount",
        regexp_replace(col("amount"), "[$,]", "").cast("decimal(12,2)")
    )
    .join(mcc_df, "mcc_code", "left")
    .join(fraud_df, "transaction_id", "left")
    .withColumn("fraud_label", col("fraud_label").cast("boolean"))
)

display(silver_transactions_df.limit(10))
silver_transactions_df.printSchema()

transaction_id,mcc_code,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,description,fraud_label
16966629,5411,2015-11-18T08:55:00.000Z,602,5979,3.80,Chip Transaction,3016,Reidsville,NC,27320.0,null,"Grocery Stores, Supermarkets",false
16966718,5499,2015-11-18T09:11:00.000Z,648,3474,0.68,Chip Transaction,14528,Clovis,NM,88101.0,null,Miscellaneous Food Stores,false
16968717,5311,2015-11-18T15:00:00.000Z,924,2210,12.05,Chip Transaction,48919,Lebanon,IN,46052.0,null,Department Stores,false
16966817,4784,2015-11-18T09:30:00.000Z,1808,4702,14.47,Online Transaction,41122,ONLINE,null,null,null,Tolls and Bridge Fees,false
16967160,5300,2015-11-18T10:29:00.000Z,1102,5010,46.99,Chip Transaction,98488,Tulsa,OK,74137.0,null,Wholesale Clubs,false
16968145,5499,2015-11-18T13:10:00.000Z,1234,22,2.09,Swipe Transaction,14528,Houston,TX,77058.0,null,Miscellaneous Food Stores,false
16967631,5311,2015-11-18T11:48:00.000Z,116,6072,46.63,Chip Transaction,99370,Le Roy,NY,14482.0,null,Department Stores,false
16967835,5814,2015-11-18T12:21:00.000Z,1054,202,12.45,Chip Transaction,45926,Gresham,OR,97080.0,null,Fast Food Restaurants,false
16965497,5499,2015-11-18T00:55:00.000Z,114,3398,49.96,Chip Transaction,59935,North Hollywood,CA,91606.0,null,Miscellaneous Food Stores,null
16966463,4111,2015-11-18T08:25:00.000Z,1117,1162,111.42,Chip Transaction,18131,Garland,TX,75043.0,null,Local and Suburban Commuter Transportation,null


root
 |-- transaction_id: long (nullable = true)
 |-- mcc_code: long (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: long (nullable = true)
 |-- card_id: long (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: long (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- errors: string (nullable = true)
 |-- description: string (nullable = true)
 |-- fraud_label: boolean (nullable = true)



In [0]:
%sql
DROP TABLE IF EXISTS silver.silver_transactions

In [0]:
silver_transactions_df.write.mode("overwrite").saveAsTable("jarvis_training.silver.silver_transactions")

In [0]:
mcc_df.write.mode("overwrite").saveAsTable("jarvis_training.silver.silver_mcc_codes")
fraud_df.write.mode("overwrite").saveAsTable("jarvis_training.silver.silver_fraud")
